### Links and Resources
| Function | Use |
|---|---|
| [`current_date`, `current_timestamp`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.current_date.html) | "Now" — captured once per query |
| [`to_date`, `to_timestamp`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.to_date.html) | Parse strings |
| [`date_format`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.date_format.html) | Format dates as strings |
| [`date_add`, `date_sub`, `add_months`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.date_add.html) | Date arithmetic |
| [`datediff`, `months_between`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.datediff.html) | Differences |
| [Datetime patterns](https://spark.apache.org/docs/latest/sql-ref-datetime-pattern.html) | `yyyy-MM-dd` etc. |

In [ ]:
import os
if not os.getenv("DATABRICKS_RUNTIME_VERSION"):
    from spark_setup import spark, display  # local run through Databricks Connect

from pyspark.sql import functions as F

# Date and Timestamp Functions

Spark distinguishes three temporal types:

- **`date`** — just `yyyy-MM-dd`, no time-of-day, no zone.
- **`timestamp`** — instant in UTC, displayed in the session time zone.
- **`timestamp_ntz`** (Spark 3.4+) — local time without zone.

Most bugs in this area come from confusing the second and third, or from format strings: Spark uses **Java DateTimeFormatter** patterns (`yyyy-MM-dd HH:mm:ss`), not strftime.

In [ ]:
df = spark.createDataFrame([
    (1, "2024-03-15",          "2024-03-15 09:30:00"),
    (2, "2024-12-31",          "2024-12-31 23:59:59"),
    (3, "2025-01-01",          "2025-01-01 00:00:01"),
    (4, "15/03/2024",          "15-03-2024 09.30.00"),  # non-standard formats
], ["id", "date_str", "ts_str"])
display(df)

## Parsing — `to_date`, `to_timestamp`

ISO 8601 strings parse without a format. Anything else needs a Java pattern.

In [ ]:
df.select(
    "date_str",
    F.to_date("date_str").alias("iso_date"),
    F.to_date("date_str", "dd/MM/yyyy").alias("eu_date"),
    F.to_timestamp("ts_str").alias("iso_ts"),
    F.to_timestamp("ts_str", "dd-MM-yyyy HH.mm.ss").alias("eu_ts"),
).show(truncate=False)

## "Now" — `current_date`, `current_timestamp`

Both are captured **once per query**, not once per row — important for reproducibility.

In [ ]:
spark.range(1).select(
    F.current_date().alias("today"),
    F.current_timestamp().alias("now"),
).show(truncate=False)

## Date arithmetic — `date_add`, `date_sub`, `add_months`, `+ INTERVAL`

`date_add(d, n)` adds n days. For other intervals, the `INTERVAL` syntax is most readable.

In [ ]:
ref = df.select(F.to_date("date_str").alias("d")).filter(F.col("d").isNotNull())

ref.select(
    "d",
    F.date_add("d", 7).alias("plus_7d"),
    F.date_sub("d", 30).alias("minus_30d"),
    F.add_months("d", 6).alias("plus_6mo"),
    F.expr("d + INTERVAL 1 YEAR").alias("plus_1y"),
).show()

## Differences — `datediff`, `months_between`

In [ ]:
ref.select(
    "d",
    F.datediff(F.current_date(), "d").alias("days_ago"),
    F.months_between(F.current_date(), "d").alias("months_ago"),
).show()

## Components — `year`, `month`, `dayofmonth`, `dayofweek`, `weekofyear`, `quarter`, `hour`, `minute`, `second`

In [ ]:
ref.select(
    "d",
    F.year("d").alias("y"),
    F.month("d").alias("m"),
    F.dayofweek("d").alias("dow"),       # 1 = Sunday
    F.dayofyear("d").alias("doy"),
    F.weekofyear("d").alias("woy"),
    F.quarter("d").alias("q"),
).show()

## Truncation — `date_trunc`, `trunc`

`date_trunc("month", ts)` zeroes out everything below the month — perfect for grouping by week/month/year.

In [ ]:
spark.range(1).select(
    F.current_timestamp().alias("now"),
    F.date_trunc("hour",   F.current_timestamp()).alias("hour"),
    F.date_trunc("day",    F.current_timestamp()).alias("day"),
    F.date_trunc("month",  F.current_timestamp()).alias("month"),
    F.trunc(F.current_date(), "MM").alias("first_of_month"),
).show(truncate=False)

## Formatting — `date_format`

Same Java pattern syntax as parsing.

In [ ]:
ref.select(
    F.date_format("d", "yyyy-MM").alias("year_month"),
    F.date_format("d", "EEEE, d MMMM yyyy").alias("long_form"),
).show(truncate=False)

## Time zones — `from_utc_timestamp`, `to_utc_timestamp`

Storage is always UTC. These functions shift the *displayed* value into a target zone, or back.

In [ ]:
spark.range(1).select(
    F.current_timestamp().alias("utc"),
    F.from_utc_timestamp(F.current_timestamp(), "Europe/Brussels").alias("brussels"),
    F.from_utc_timestamp(F.current_timestamp(), "America/New_York").alias("ny"),
).show(truncate=False)

## Unix timestamps

`unix_timestamp()` → bigint seconds since epoch. `from_unixtime` reverses it.

In [ ]:
ref.select(
    "d",
    F.unix_timestamp("d").alias("epoch_seconds"),
    F.from_unixtime(F.unix_timestamp("d")).alias("round_trip"),
).show()

## Takeaways

- ISO 8601 strings auto-parse; everything else needs a Java pattern (NOT strftime).
- `current_date()`/`current_timestamp()` are query-stable — same value for every row in one query.
- Group by `date_trunc("month", ...)` for monthly rollups.
- Time zone conversions are display-time only; the underlying timestamp is always a UTC instant.
- `dayofweek` is **1 = Sunday** (US convention), not Monday.